# 6 · Create Linear Probing Sets
Builds balanced particle subsets per annotation for linear-probing training and evaluation.

**Two variants:**
- `ue_train` — both train and eval sampled from the pre-training split.
- `train_on_eval` — train from pre-training split, eval from the held-out split.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
ANN_PARQUET = "./metadata/parquet_files/df_ann.parquet"
OUT_DIR     = "./metadata/parquet_files"   # must contain train/ and eval/ sub-dirs
EVAL_FRAC   = 0.15
SEED_SPLIT  = 37   # seed for experiment-level train/eval split
SEED_SAMPLE = 37   # seed for within-bin sampling

In [2]:
import numpy as np
import pandas as pd

## Configuration tables

In [3]:
# Bin boundaries: {annotation: (bin_width, start, end)}
BIN_DICT = {
    'MolecularWeight_kDa':      (20,    0,      1000),
    'DefocusU':                  (10000, -5000,  40000),
    'DefocusV':                  (10000, -5000,  40000),
    'DefocusAngle':              (30,    -105,   105),
    '2DAlignmentClassESS':       (1,     0.5,    5),
    'ClassECA':                  (0.5,   0.75,   3),
    'ClassResolution':           (3,     1.5,    26),
    'EMD_Resolution':            (0.5,   1.25,   8),
    'ReconstructionResolution':  (0.5,   1.25,   8),
    'ImagePixelSize':            (0.25,  0.875,  3),
}

# Max samples per bin per annotation (training set)
NUM_OF_PICKS = {
    'ReconstructionResolution': 67_000,
    'EMD_Resolution':           60_000,
    'MolecularWeight_kDa':      20_000,
    'ImagePixelSize':           90_000,
    'ClassResolution':          90_000,
    'ClassECA':                160_000,
    '2DAlignmentClassESS':     160_000,
    'DefocusAngle':            115_000,
    'DefocusU':                160_000,
    'DefocusV':                160_000,
    'ProteinClass':             60_000,
    'Symmetry':                 80_000,
    'FileName':                  3_000,
}

ANNOTATIONS_UE_TRAIN = [
    'EMD_Resolution', 'ReconstructionResolution', 'MolecularWeight_kDa',
    'ImagePixelSize', 'ClassResolution', 'ClassECA', '2DAlignmentClassESS',
    'DefocusAngle', 'DefocusU', 'DefocusV', 'ProteinClass', 'Symmetry', 'FileName',
]
ANNOTATIONS_TRAIN_ON_EVAL = [a for a in ANNOTATIONS_UE_TRAIN if a != 'FileName']

OUTPUT_COLS = ['FileIndex', 'FileName', 'FileParticleIndex',
               'BinIndex', 'BinCenter', 'AnnotationValue']

## Helper functions

In [4]:
def get_train_eval_masks(df_ann, eval_frac, seed):
    np.random.seed(seed)
    unique_exp     = np.unique(df_ann['FileName'].values)
    n              = len(unique_exp)
    n_eval         = int(n * eval_frac)
    unique_exp_idx = np.arange(n)
    np.random.shuffle(unique_exp_idx)
    eval_idxs  = np.sort(unique_exp_idx[:n_eval])
    train_idxs = np.sort(unique_exp_idx[n_eval:])
    return (np.isin(df_ann['FileIndex'].values, train_idxs),
            np.isin(df_ann['FileIndex'].values, eval_idxs))


def compute_bins(particles, ann):
    """Return (bin_centers, bin_idx, counts, categorical_map_or_None)."""
    if ann in ('ProteinClass', 'Symmetry', 'FileName'):
        unique_cats, bin_idx, counts = np.unique(
            particles[ann], return_inverse=True, return_counts=True)
        return np.arange(len(unique_cats)), bin_idx, counts, {c:i for i,c in enumerate(unique_cats)}

    values = (particles['ImagePixelSize'] * particles['ImageSize'] / 224
              if ann == 'ImagePixelSize' else particles[ann].astype(float))
    bin_dist, start, end = BIN_DICT.get(ann, (0.5, 1.25, 8))
    bin_edges   = np.arange(start, end + bin_dist, bin_dist)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_idx     = np.digitize(values, bin_edges, right=True) - 1
    bin_idx[bin_idx == -1] = bin_idx.max()
    return bin_centers, bin_idx, np.bincount(bin_idx), None


def sample_indices(bin_centers, bin_idx, counts, n_picks, seed):
    np.random.seed(seed)
    selected = []
    for bci in range(len(bin_centers)):
        if not counts[bci]:
            continue
        idx = np.where(bin_idx == bci)[0]
        np.random.shuffle(idx)
        selected.append(idx[:n_picks])
    return np.sort(np.concatenate(selected))


def build_df(particles, pos, bin_idx, bin_centers, ann):
    df = particles.iloc[pos].copy()
    df['BinIndex']        = bin_idx[pos]
    df['BinCenter']       = bin_centers[df['BinIndex']]
    df['AnnotationValue'] = particles[ann].iloc[pos].values
    return df[OUTPUT_COLS]


def remap_bin_index(df):
    df = df.copy()
    _, inv = np.unique(df['BinIndex'], return_inverse=True)
    df['BinIndex'] = inv
    return df, dict(zip(df['BinCenter'], df['BinIndex']))


def apply_bin_mapping(df, mapping):
    df = df.copy()
    df['BinIndex'] = df['BinCenter'].map(mapping)
    df = df.dropna(subset=['BinIndex'])
    df['BinIndex'] = df['BinIndex'].astype(int)
    return df

## 1 · Load data and create split masks

In [5]:
df_ann = pd.read_parquet(ANN_PARQUET)
train_mask, eval_mask = get_train_eval_masks(df_ann, EVAL_FRAC, SEED_SPLIT)
train_particles = df_ann.loc[train_mask]
eval_particles  = df_ann.loc[eval_mask]
print(f"Train: {train_mask.sum():,} particles | Eval: {eval_mask.sum():,} particles")

Train: 33,120,888 particles | Eval: 4,502,235 particles


## 2 · Build `ue_train` datasets
Both train and eval sampled from the **pre-training split** (non-overlapping within each bin).

In [6]:
NUM_OF_PICKS_UE = {**NUM_OF_PICKS, 'ProteinClass': 50_000}

for ann in ANNOTATIONS_UE_TRAIN:
    bin_centers, bin_idx, counts, _ = compute_bins(train_particles, ann)
    n          = NUM_OF_PICKS_UE[ann]
    np.random.seed(SEED_SAMPLE)
    sel_train, sel_eval = [], []

    for bci in range(len(bin_centers)):
        if not counts[bci]:
            continue
        idx = np.where(bin_idx == bci)[0]
        np.random.shuffle(idx)
        sel_train.append(idx[:n])
        sel_eval.append(idx[n:n + int(0.2 * n)])

    for split, selected in [('train', sel_train), ('eval', sel_eval)]:
        pos    = np.sort(np.concatenate(selected))
        df_out = build_df(train_particles, pos, bin_idx, bin_centers, ann)
        if split == 'train':
            df_out, bincenter_map = remap_bin_index(df_out)
        else:
            df_out = apply_bin_mapping(df_out, bincenter_map)
        path = f"{OUT_DIR}/{split}/{ann}_ue_train.parquet"
        df_out.to_parquet(path, index=False)

    print(f"  {ann} done")

print("ue_train complete")

  EMD_Resolution done
  ReconstructionResolution done
  MolecularWeight_kDa done
  ImagePixelSize done
  ClassResolution done
  ClassECA done
  2DAlignmentClassESS done
  DefocusAngle done
  DefocusU done
  DefocusV done
  ProteinClass done
  Symmetry done
  FileName done
ue_train complete


## 3 · Build `train_on_eval` datasets
Train from the **pre-training split**, eval from the **held-out split**.

In [7]:
for ann in ANNOTATIONS_TRAIN_ON_EVAL:
    # ── Train ──
    bin_centers, bin_idx, counts, cat_map = compute_bins(train_particles, ann)
    pos     = sample_indices(bin_centers, bin_idx, counts, NUM_OF_PICKS[ann], SEED_SAMPLE)
    df_train, bincenter_map = remap_bin_index(build_df(train_particles, pos, bin_idx, bin_centers, ann))
    df_train.to_parquet(f"{OUT_DIR}/train/{ann}_ue_train_on_eval.parquet", index=False)

    # ── Eval ──
    if cat_map is not None:
        eval_bin_idx     = np.array(eval_particles[ann].map(cat_map))
        eval_bin_centers = np.arange(max(cat_map.values()) + 1)
        valid            = ~np.isnan(eval_bin_idx.astype(float))
        eval_counts      = np.bincount(eval_bin_idx[valid].astype(int), minlength=len(eval_bin_centers))
    else:
        eval_bin_centers, eval_bin_idx, eval_counts, _ = compute_bins(eval_particles, ann)

    eval_pos = sample_indices(eval_bin_centers, eval_bin_idx, eval_counts,
                              int(0.2 * NUM_OF_PICKS[ann]), SEED_SAMPLE)
    df_eval  = apply_bin_mapping(
        build_df(eval_particles, eval_pos, eval_bin_idx, eval_bin_centers, ann),
        bincenter_map
    )
    df_eval.to_parquet(f"{OUT_DIR}/eval/{ann}_ue_train_on_eval.parquet", index=False)
    print(f"  {ann} done")

print("train_on_eval complete")

  EMD_Resolution done
  ReconstructionResolution done
  MolecularWeight_kDa done
  ImagePixelSize done
  ClassResolution done
  ClassECA done
  2DAlignmentClassESS done
  DefocusAngle done
  DefocusU done
  DefocusV done
  ProteinClass done
  Symmetry done
train_on_eval complete
